In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# ── Cell 1: Colab environment bootstrap ──────────────────────────
import os, sys, subprocess

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/eeg_pipeline'
DATA_DIR   = os.path.join(DRIVE_ROOT, 'data/raw/eegdenoisenet/data')

# 2. Clone repo (skip if already cloned)
REPO_URL = 'https://github.com/Yago-Velazquez/eeg_pipeline'
REPO_DIR = '/content/eeg_pipeline'

if not os.path.exists(REPO_DIR):
    result = subprocess.run(
        ['git', 'clone', REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout)
    print(result.stderr)  # ← this will show exactly why it failed
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# 3. Install dependencies
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'])

# 4. Symlink EEGDenoiseNet data
LINK_TARGET = os.path.join(REPO_DIR, 'data/raw/eegdenoisenet/data')
if not os.path.exists(LINK_TARGET):
    os.makedirs(os.path.dirname(LINK_TARGET), exist_ok=True)
    os.symlink(DATA_DIR, LINK_TARGET)

# 5. Symlink BCR CSV
BCR_CSV_SRC  = os.path.join(DRIVE_ROOT, 'data/raw/Bad_channels_for_ML.csv')
BCR_CSV_LINK = os.path.join(REPO_DIR, 'data/raw/Bad_channels_for_ML.csv')
if not os.path.exists(BCR_CSV_LINK):
    os.makedirs(os.path.dirname(BCR_CSV_LINK), exist_ok=True)
    os.symlink(BCR_CSV_SRC, BCR_CSV_LINK)

print("✓ Drive mounted")
print(f"✓ Repo at: {REPO_DIR}")
print(f"✓ EEGDenoiseNet linked: {os.path.exists(LINK_TARGET)}")
print(f"✓ BCR CSV linked: {os.path.exists(BCR_CSV_LINK)}")
print(f"✓ Python path includes repo: {REPO_DIR in sys.path}")

In [ ]:
# ── Cell 2: Verify GPU + imports ─────────────────────────────────
import torch
import numpy as np

# GPU check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Core imports from our modules
from data.preprocessing import bandpass_filter, zscore_normalize, segment_signal
from data.reconstruction import reconstruct_signal
from evaluate.metrics import compute_auprc, snr_improvement, pearson_r

print("✓ All module imports OK")

# Data file check
import os
files = [
    'data/raw/eegdenoisenet/data/EEG_all_epochs.npy',
    'data/raw/eegdenoisenet/data/EOG_all_epochs.npy',
    'data/raw/eegdenoisenet/data/EEG_all_epochs_512hz.npy',
    'data/raw/eegdenoisenet/data/EMG_all_epochs_512hz.npy',
    'data/raw/Bad_channels_for_ML.csv',
]
for f in files:
    exists = os.path.exists(f)
    print(f"{'✓' if exists else '✗'} {f}")

In [ ]:
# ── Cell 3: Integration smoke test ───────────────────────────────
import numpy as np
import pandas as pd

# ── 1. BCR: load data, decode session IDs ────────────────────────
print("[1/4] Loading BCR data...")
BCR_CSV = 'data/raw/Bad_channels_for_ML.csv'
try:
    df = pd.read_csv(BCR_CSV)
    # Session format: site-subject-visit (e.g. '1-103-1')
    parts = df['Session'].str.split('-', expand=True)
    df['session_group'] = parts[0].astype(int)   # site (1–4)
    df['subject_id']    = parts[1].astype(int)   # subject (e.g. 103)
    df['visit']         = parts[2].astype(int)   # visit number
    df['is_bad']        = (df['Bad (score)'] >= 2).astype(int)
    n_subjects = df['subject_id'].nunique()
    bad_rate   = df['is_bad'].mean()
    print(f"  ✓ BCR: {len(df):,} rows, {n_subjects} subjects, bad_rate={bad_rate:.3f}")
except FileNotFoundError:
    print("  ⚠ BCR CSV not found — using dummy data")
    n_subjects = 43; bad_rate = 0.039
    print(f"  (dummy) subjects={n_subjects}, bad_rate={bad_rate}")

# ── 2. EEGDenoiseNet: load clean + EOG arrays ────────────────────
print("[2/4] Loading EEGDenoiseNet...")
eeg_clean = np.load('data/raw/eegdenoisenet/data/EEG_all_epochs.npy')
eog_noise = np.load('data/raw/eegdenoisenet/data/EOG_all_epochs.npy')
print(f"  ✓ EEG clean: {eeg_clean.shape}, EOG noise: {eog_noise.shape}")

# ── 3. Preprocessing pipeline ────────────────────────────────────
print("[3/4] Running preprocessing pipeline...")
from data.preprocessing import bandpass_filter, zscore_normalize, segment_signal

sample = eeg_clean[0].astype(np.float32)    # shape: (256,)
filtered  = bandpass_filter(sample, fs=256)
normalized = zscore_normalize(filtered)
segments  = segment_signal(normalized, window=256, overlap=0.5)
print(f"  ✓ Input: {sample.shape} → Filtered → Normalized → Segments: {segments.shape}")

# ── 4. Metrics with dummy values ─────────────────────────────────
print("[4/4] Checking metrics module...")
from evaluate.metrics import compute_auprc, pearson_r

# Dummy BCR metrics (random-ish labels)
y_true = np.array([0]*100 + [1]*4)
y_prob = np.random.rand(104)
auprc = compute_auprc(y_true, y_prob)
print(f"  ✓ AUPRC on dummy data: {auprc:.4f}  (random baseline ≈ 0.039)")

# Dummy denoiser metrics
clean_seg  = segments[0].flatten()
noisy_seg  = clean_seg + np.random.normal(0, 0.2, clean_seg.shape)
r = pearson_r(clean_seg, clean_seg)            # perfect = 1.0
print(f"  ✓ Pearson r (perfect): {r:.4f}")

print("\n✅ Integration smoke test PASSED")